In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType

spark.conf.set(
    "fs.azure.account.key.datalake0501.dfs.core.windows.net",
    "YOUR_STORAGE_ACCOUNT_KEY_HERE"  # Replace with your key
)

df_bronze = spark.read.csv(
    'abfss://bronze@datalake0501.dfs.core.windows.net/ecommerce/raw/ecommerce_dataset_1m.csv',
    header=True,
    inferSchema=False  # explicit casting below
)

# Explicit type casting
df_bronze = df_bronze \
    .withColumn("unit_price_usd", F.col("unit_price_usd").cast(DecimalType(10,2))) \
    .withColumn("total_price_usd", F.col("total_price_usd").cast(DecimalType(10,2))) \
    .withColumn("profit_usd", F.col("profit_usd").cast(DecimalType(10,2))) \
    .withColumn("cost_usd", F.col("cost_usd").cast(DecimalType(10,2))) \
    .withColumn("discount_amount_usd", F.col("discount_amount_usd").cast(DecimalType(10,2))) \
    .withColumn("tax_usd", F.col("tax_usd").cast(DecimalType(10,2))) \
    .withColumn("shipping_cost_usd", F.col("shipping_cost_usd").cast(DecimalType(10,2))) \
    .withColumn("age", F.col("age").cast(IntegerType())) \
    .withColumn("quantity", F.col("quantity").cast(IntegerType())) \
    .withColumn("discount_percent", F.col("discount_percent").cast(IntegerType())) \
    .withColumn("fraud_risk_score", F.col("fraud_risk_score").cast(DecimalType(5,2))) \
    .withColumn("customer_loyalty_score", F.col("customer_loyalty_score").cast(DecimalType(5,2))) \
    .withColumn("account_creation_date", F.to_date("account_creation_date")) \
    .withColumn("order_date", F.to_timestamp("order_date"))

print(f"Bronze records: {df_bronze.count()}")

In [0]:
df_fact_orders = df_bronze \
    .dropDuplicates(['order_id']) \
    .filter(F.col('order_id').isNotNull()) \
    .filter(F.col('customer_id').isNotNull()) \
    .filter(F.col('total_price_usd').cast(DecimalType(10,2)) > 0) \
    .filter(F.col('quantity').cast(IntegerType()) > 0) \
    .withColumn('order_year', F.year('order_date')) \
    .withColumn('order_month', F.month('order_date')) \
    .withColumn('order_week', F.weekofyear('order_date')) \
    .withColumn('is_weekend', F.when(F.col('is_weekend') == 'Yes', True).otherwise(False)) \
    .withColumn('coupon_used', F.when(F.col('coupon_used') == 'Yes', True).otherwise(False)) \
    .withColumn('installment_plan', F.when(F.col('installment_plan') == 'Yes', True).otherwise(False)) \
    .withColumn('abandoned_cart_before', F.when(F.col('abandoned_cart_before') == 'Yes', True).otherwise(False)) \
    .withColumn('support_ticket_created', F.when(F.col('support_ticket_created') == 'Yes', True).otherwise(False)) \
    .withColumn('profit_margin', 
        F.round((F.col('profit_usd') / F.col('total_price_usd')) * 100, 2)) \
    .withColumn('high_value_order_flag', 
        F.when(F.col('total_price_usd') > 500, True).otherwise(False)) \
    .withColumn('fraud_flag', 
        F.when(F.col('fraud_risk_score') > 75, True).otherwise(False)) \
    .withColumn('return_reason', 
        F.when(F.col('return_reason').isNull(), 'No Return').otherwise(F.col('return_reason'))) \
    .withColumn('ingestion_timestamp', F.current_timestamp()) \
    .select(
        'order_id', 'order_date', 'order_year', 'order_month', 'order_week',
        'customer_id', 'product_id', 'quantity', 'unit_price_usd',
        'discount_percent', 'discount_amount_usd', 'total_price_usd',
        'cost_usd', 'profit_usd', 'tax_usd', 'shipping_cost_usd',
        'profit_margin', 'high_value_order_flag', 'fraud_flag',
        'order_status', 'return_reason', 'payment_method', 'payment_status',
        'shipping_method', 'delivery_days', 'delivery_status',
        'is_weekend', 'coupon_used', 'coupon_code', 'campaign_source',
        'fraud_risk_score', 'order_priority', 'support_ticket_created',
        'ingestion_timestamp'
    )

df_fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .save("abfss://silver@datalake0501.dfs.core.windows.net/fact_orders/")

print(f"fact_orders written: {df_fact_orders.count()}")

In [0]:
df_dim_customer = df_bronze.select(
    'customer_id', 'customer_name', 'gender', 'age',
    'customer_segment', 'country', 'city',
    'customer_loyalty_score', 'total_orders_by_customer',
    'account_creation_date'
).dropDuplicates(['customer_id']) \
 .filter(F.col('customer_id').isNotNull()) \
 .fillna({
     'age': 0,
     'customer_loyalty_score': 0.0,
     'total_orders_by_customer': 0,
     'gender': 'Unknown',
     'customer_segment': 'Unknown'
 })

df_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("country") \
    .save("abfss://silver@datalake0501.dfs.core.windows.net/dim_customer/")

print(f"dim_customer written: {df_dim_customer.count()}")

In [0]:
df_dim_product = df_bronze.select(
    'product_id', 'product_name', 'category', 'sub_category',
    'brand', 'product_rating_avg', 'product_reviews_count',
    'stock_quantity', 'unit_price_usd'
).dropDuplicates(['product_id']) \
 .filter(F.col('product_id').isNotNull()) \
 .fillna({
     'brand': 'Unknown',
     'sub_category': 'Unknown',
     'product_rating_avg': 0.0,
     'product_reviews_count': 0,
     'stock_quantity': 0
 })

df_dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("category") \
    .save("abfss://silver@datalake0501.dfs.core.windows.net/dim_product/")

print(f"dim_product written: {df_dim_product.count()}")

In [0]:
# Verify star schema
fact = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/fact_orders/"
)
dim_cust = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_customer/"
)
dim_prod = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_product/"
)

print(f"fact_orders:  {fact.count()} rows, {len(fact.columns)} cols")
print(f"dim_customer: {dim_cust.count()} rows, {len(dim_cust.columns)} cols")
print(f"dim_product:  {dim_prod.count()} rows, {len(dim_prod.columns)} cols")

# Quick sanity check on business columns
fact.select(
    'order_id', 'profit_margin', 
    'high_value_order_flag', 'fraud_flag', 
    'order_week'
).show(3)

In [0]:
fact_orders = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/ecommerce/fact_orders/"
)

In [0]:
display(dbutils.fs.ls("abfss://silver@datalake0501.dfs.core.windows.net/"))

In [0]:
display(dbutils.fs.ls("abfss://silver@datalake0501.dfs.core.windows.net/ecommerce/"))

In [0]:
fact_orders = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/fact_orders/"
)

In [0]:
dim_customer = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_customer/"
)

dim_product = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_product/"
)

In [0]:
fact_orders = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/fact_orders/"
)
dim_customer = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_customer/"
)
dim_product = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_product/"
)

print(f"fact_orders:  {fact_orders.count()} rows, {len(fact_orders.columns)} cols")
print(f"dim_customer: {dim_customer.count()} rows, {len(dim_customer.columns)} cols")
print(f"dim_product:  {dim_product.count()} rows, {len(dim_product.columns)} cols")

fact_orders.select(
    'order_id', 'profit_margin',
    'high_value_order_flag', 'fraud_flag',
    'order_week'
).show(3)

In [0]:
from pyspark.sql import functions as F

# Check 1 - Duplicate orders
dups = fact_orders.groupBy("order_id").count().filter(F.col("count") > 1).count()
print(f"Duplicate orders: {dups}")

# Check 2 - Fraud flag distribution
print("\nFraud flag distribution:")
fact_orders.groupBy("fraud_flag").count().show()

# Check 3 - High value flag distribution
print("High value order distribution:")
fact_orders.groupBy("high_value_order_flag").count().show()

# Check 4 - Price sanity
invalid_prices = fact_orders.filter(F.col("unit_price_usd") <= 0).count()
print(f"Invalid prices: {invalid_prices}")

# Check 5 - Null check on key columns
print("Null counts on key columns:")
fact_orders.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['order_id', 'customer_id', 'product_id', 'total_price_usd', 'profit_margin']
]).show()

# Check 6 - Partition distribution
print("Orders by year:")
fact_orders.groupBy("order_year").count().orderBy("order_year").show()